# Exercise: Classify Sea Animals with Transfer Learning

[Caltech-101](https://data.caltech.edu/records/mzrjq-6wc02) is a collection of about 9,000 photos of objects in 101 categories, collected at the California Institute of Technology in 2003. In this exercise you use five of its categories, all animals that live in water: **crabs, crayfish, dolphins, lobsters and starfish** (all from the sea except the crayfish, which lives in fresh water). Each category has only 41 to 86 photos, far too few to train a CNN from scratch, which makes it a typical case for transfer learning. Three of the five are crustaceans with claws and segmented bodies, so telling them apart is the hard part.

You apply the workflow of notebooks 01 and 02: test a pretrained network as it is, extract features and train a dense head on them, and train a frozen base with augmentation. You compare everything on the validation set and evaluate your chosen model once on the test set. The tasks describe what to build and what to report; the code is yours to write.

## Learning Objectives

By the end of this exercise, you should be able to:

- Load a small image dataset from folders and inspect its classes.
- Train a dense classification head on the features of a frozen pretrained base.
- Train a frozen base with data augmentation.
- Compare the models on the validation set.
- Evaluate the chosen model once on the test set and identify the classes it confuses.

## Setup

In [ ]:
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from keras.applications import mobilenet_v2
from sklearn.metrics import ConfusionMatrixDisplay

keras.utils.set_random_seed(42)
IMAGE_SIZE = (160, 160)

# One plot style for the whole notebook
plt.rcParams.update(
    {"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3}
)

## Step 1: The Data

The repository contains the photos in `data/sea_animals/`, already split per category into 60 % training, 20 % validation and 20 % test photos, in the folder layout `image_dataset_from_directory` expects: `data/sea_animals/<split>/<class>/`. The cell below counts them.

In [ ]:
CLASSES = ["crab", "crayfish", "dolphin", "lobster", "starfish"]
sea_animals = Path("data") / "sea_animals"

pd.DataFrame(
    {split: {name: len(list((sea_animals / split / name).glob("*.jpg"))) for name in CLASSES}
     for split in ["train", "validation", "test"]}
)

## Step 2: Load the Images

- Load the three splits with `image_dataset_from_directory` from `data/sea_animals/<split>`, with `IMAGE_SIZE`, a batch size of 32 and integer labels. Shuffle the training set (with `seed=42`), not the validation and test sets.
- Plot one example image of each class with its class name.

Caltech-101 contains a few greyscale photos; `image_dataset_from_directory` converts them to three colour channels by default.

## Step 3: A Pretrained Network as It Is

Classify the first validation photo of each class (in sorted file order) with the complete MobileNetV2 (`weights="imagenet"`, images of 224 × 224 pixels, `preprocess_input` and `decode_predictions` from `mobilenet_v2`), and print its top-3 ImageNet classes.

Which animals does MobileNetV2 recognise? ImageNet has classes for crabs, crayfish, lobsters and starfish, but not for dolphins: what does the network answer for the animals it does not recognise?

*Your answers:*

## Step 4: Features and a Dense Head

- Create the MobileNetV2 base without its classifier, with `pooling="avg"` and `IMAGE_SIZE`, and freeze it.
- Extract the feature vectors and labels of the training, validation and test sets. Remember to use an **unshuffled** copy of the training set.
- Train the dense head of notebook 02 on the training features: `Dropout(0.2)` and a softmax output layer for the five classes, with Adam (learning rate 0.001) for 30 epochs. Set the random seed to 42 first.

Create a dictionary `results` and store the validation accuracy of every model in it, starting with this one.

How does the validation accuracy compare with the dense head on the bean leaf features in notebook 02 (0.910), which had about eight times as many training images per class?

*Your answers:*

## Step 5: Frozen Base with Data Augmentation

Build a `keras.Sequential` model: an augmentation block (`RandomFlip("horizontal")`, `RandomRotation(0.1)`, `RandomZoom(0.1)`), the MobileNetV2 scaling as a `Rescaling(scale=1 / 127.5, offset=-1)` layer, the frozen base, `Dropout(0.2)` and a softmax output layer for the five classes.

Compile it with Adam (learning rate 0.001) and the sparse categorical cross-entropy, train it for 10 epochs on the image datasets, plot the learning curves and store its validation accuracy.

Is a horizontal flip a sensible augmentation for these photos? Would a vertical flip be?

*Your answers:*

## Step 6: Compare and Choose

Show the validation accuracies of both models in one table, sorted from best to worst, and choose your final model. The validation set has only 67 images: how many images does a difference of 0.015 correspond to? Take this into account in your choice.

*Your answers:*

## Step 7: Evaluate Once on the Test Set

Evaluate your chosen model on the test set and plot its confusion matrix with the class names. Which classes does it confuse, and why could that be?

*Your answers:*

## Reflection

Answer these questions in the Markdown cell below.

1. The dense head on the features of a frozen base has only one trainable layer. Why can it perform as well as the model with augmentation on this data?
2. With data augmentation, the features cannot be computed once in advance. Why not, and what do you gain and lose with each approach?
3. Your training set has fewer than 60 images per class. What would you expect from a CNN trained from scratch on it, and why?

*Your answers:*

## References

- Fei-Fei, L., Fergus, R. and Perona, P. (2004). Learning generative visual models from few training examples. [Caltech-101 dataset](https://data.caltech.edu/records/mzrjq-6wc02), CC BY 4.0.
- Keras guide: [Transfer learning and fine-tuning](https://keras.io/guides/transfer_learning/).